# Visualization 3: Gender Effects on Name Popularity

**Design**: Dual area chart M/F — two superposed area charts for a mixed-gender name.
The crossing zone indicates a popularity inversion between boys and girls.

- X-axis: year
- Y-axis: number of births
- Two areas: Filles (F) and Garçons (M)
- Interactive dropdown to select any popular mixed-gender name

In [ ]:
import pandas as pd
import altair as alt

## Load and prepare data

In [ ]:
df = pd.read_csv('dpt2020.csv', sep=';', dtype={'annais': str, 'dpt': str})

df = df[(df['preusuel'] != '_PRENOMS_RARES') & (df['annais'] != 'XXXX')]
df['annais'] = df['annais'].astype(int)

# Aggregate nationally
national = df.groupby(['preusuel', 'sexe', 'annais'], as_index=False)['nombre'].sum()
national.head()

## Find popular mixed-gender names and compute the best default

In [ ]:
# Names given to both sexes
sexes_per_name = national.groupby('preusuel')['sexe'].nunique()
mixed_names = sexes_per_name[sexes_per_name == 2].index

mixed_df = national[national['preusuel'].isin(mixed_names)]

# Keep names with enough births to be meaningful
total_per_name = mixed_df.groupby('preusuel')['nombre'].sum()
popular_mixed = sorted(total_per_name[total_per_name >= 5000].index.tolist())

# Default: most balanced name by M/F ratio (closest to 50/50), among popular ones
pivot = (
    mixed_df[mixed_df['preusuel'].isin(popular_mixed)]
    .groupby(['preusuel', 'sexe'])['nombre'].sum()
    .unstack(fill_value=0)
)
pivot.columns = ['M', 'F']
pivot['balance'] = pivot.min(axis=1) / pivot.sum(axis=1)

# Among top-100 most popular, pick the most balanced
top100 = total_per_name[popular_mixed].nlargest(100).index
default_name = pivot.loc[top100, 'balance'].idxmax()

print(f'Default name: {default_name}')
print(f'Popular mixed-gender names ({len(popular_mixed)} total):', popular_mixed[:10], '...')

## Build the dual area chart

In [ ]:
viz_df = mixed_df[mixed_df['preusuel'].isin(popular_mixed)].copy()
viz_df['gender'] = viz_df['sexe'].map({1: 'Garçons', 2: 'Filles'})

name_selector = alt.binding_select(options=popular_mixed, name='Prénom : ')
name_param = alt.param(name='selected_name', value=default_name, bind=name_selector)

color_scale = alt.Scale(
    domain=['Garçons', 'Filles'],
    range=['#4C9BE8', '#E8748C']
)

area = (
    alt.Chart(viz_df)
    .mark_area(opacity=0.55, interpolate='monotone')
    .encode(
        x=alt.X('annais:Q', title='Année', axis=alt.Axis(format='d')),
        y=alt.Y('nombre:Q', title='Nombre de naissances'),
        color=alt.Color('gender:N', scale=color_scale, title='Sexe'),
        tooltip=[
            alt.Tooltip('annais:Q', title='Année'),
            alt.Tooltip('gender:N', title='Sexe'),
            alt.Tooltip('nombre:Q', title='Naissances', format=','),
        ]
    )
    .transform_filter(alt.datum.preusuel == name_param)
    .add_params(name_param)
    .properties(
        width=750,
        height=380,
        title=alt.TitleParams(
            text='Popularité M/F d\'un prénom mixte au fil du temps',
            subtitle='La zone de croisement indique une inversion de popularité entre les sexes',
            fontSize=16,
            subtitleFontSize=12,
        )
    )
)

area

## Difference chart: highlights crossover moments

In [ ]:
pivot_time = (
    viz_df.pivot_table(
        index=['preusuel', 'annais'], columns='gender', values='nombre', fill_value=0
    )
    .reset_index()
)
pivot_time.columns.name = None
pivot_time['diff'] = pivot_time.get('Filles', 0) - pivot_time.get('Garçons', 0)

diff_line = (
    alt.Chart(pivot_time)
    .mark_line(color='black', strokeDash=[4, 4], strokeWidth=1.2, opacity=0.5)
    .encode(
        x=alt.X('annais:Q', title='Année', axis=alt.Axis(format='d')),
        y=alt.Y('diff:Q', title='Filles − Garçons'),
        tooltip=[
            alt.Tooltip('annais:Q', title='Année'),
            alt.Tooltip('diff:Q', title='Filles − Garçons', format=','),
        ]
    )
    .transform_filter(alt.datum.preusuel == name_param)
    .add_params(name_param)
    .properties(width=750, height=120, title='Différence Filles − Garçons (0 = parité)')
)

zero_rule = (
    alt.Chart(pd.DataFrame({'y': [0]}))
    .mark_rule(color='red', strokeDash=[3, 3], opacity=0.6)
    .encode(y='y:Q')
)

final_chart = (
    alt.vconcat(area, diff_line + zero_rule)
    .configure_legend(orient='top-right', labelFontSize=13, titleFontSize=13)
    .configure_axis(labelFontSize=12, titleFontSize=13)
)

final_chart

## Save to HTML

In [ ]:
final_chart.save('viz3_gender.html')
print('Saved to viz3_gender.html')